<a href="https://colab.research.google.com/github/pablofernandezrubal/tfg-embeddings/blob/main/01_visualization_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap

In [ ]:
#Triger authentication flow
ee.Authenticate()

# Initialize the library
ee.Initialize(project='gen-lang-client-0148391940')

In [ ]:
!pip install osmnx

In [ ]:
import osmnx as ox

# Obtiene la geometría de Burela
gdf = ox.geocode_to_gdf("Burela, Lugo, Spain")

# Extraer el MultiPolygon
polygon = gdf.geometry.iloc[0]
print(polygon)

In [ ]:
coords = list(polygon.exterior.coords)
print(coords)

In [ ]:
geometry = ee.Geometry.Polygon(coords)

In [ ]:
# geoJSON = {
#   "type": "FeatureCollection",
#   "features": [
#     {
#       "type": "Feature",
#       "properties": {},
#       "geometry": {
#         "coordinates": [
#           [
#             [
#               -7.408043908593299,
#               43.45464557006821
#             ],
#             [
#               -7.408043908593299,
#               43.413658496069786
#             ],
#             [
#               -7.328783243763866,
#               43.413658496069786
#             ],
#             [
#               -7.328783243763866,
#               43.45464557006821
#             ],
#             [
#               -7.408043908593299,
#               43.45464557006821
#             ]
#           ]
#         ],
#         "type": "Polygon"
#       }
#     }
#   ]
# }

# coords = geoJSON["features"][0]["geometry"]["coordinates"]
# geometry = ee.Geometry.Polygon(coords)


In [ ]:
mapa = geemap.Map()
mapa.setOptions('SATELLITE')
mapa.centerObject(geometry, 12)

In [ ]:
embeddings = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')

In [ ]:
startDate = ee.Date.fromYMD(2024, 1, 1)
endDate = startDate.advance(1, 'year')

filter_embeddings = embeddings.filter(ee.Filter.date(startDate, endDate)).filter(ee.Filter.bounds(geometry))

In [ ]:
embeddings_mosaic = filter_embeddings.mosaic()

In [ ]:
print(embeddings_mosaic.getInfo())

In [ ]:
mapa.addLayer(embeddings_mosaic.clip(geometry), {}, 'embeddings')

In [ ]:
mapa.addLayer(geometry, {}, 'geometry')

In [ ]:
vis_params = {
    'min': -0.3,
    'max': 0.3,
    'bands': ['A11', 'A22', 'A33'] #podemos poner las 3 bandas que queramos
}

In [ ]:
mapa.addLayer(embeddings_mosaic.clip(geometry), vis_params, '3 bandas')

# Unsupervised clustering

In [ ]:
# extraemos puntos para el entrenamiento

nSamples = 1000
training = embeddings_mosaic.sample(
    region = geometry,    # que sean puntos dentro de burela
    scale = 10,           # resolución nominal en metros
    numPixels = nSamples,     # número total de samples/pixels
    seed = 100,        # semilla para reproducibilidad
    geometries = True,     #
    tileScale = 8     # para subidivir la tarea
)
print(training.first().getInfo())

In [ ]:
def getClusters(embeddings_image, training_data, nClusters: int):
  clusterer = ee.Clusterer.wekaKMeans(nClusters=nClusters, distanceFunction="cosin").train(training_data)
  clustered = embeddings_image.cluster(clusterer)
  return clustered

In [ ]:
embeddings_clustered10 = getClusters(embeddings_mosaic, training, 10)
mapa.addLayer(embeddings_clustered10.randomVisualizer().clip(geometry), {}, '10 clusters')

In [ ]:
def CascadeClusters(embeddings_image, training_data, minClusters, maxClusters):
  clusterer = ee.Clusterer.wekaCascadeKMeans(minClusters=minClusters, maxClusters=maxClusters, distanceFunction="cosin").train(training_data)
  clustered = embeddings_image.cluster(clusterer)
  return clustered

In [ ]:
embeddings_cascadeCluster = CascadeClusters(embeddings_mosaic, training, 4, 11)
mapa.addLayer(embeddings_cascadeCluster.randomVisualizer().clip(geometry), {}, 'Cascade Clustering')

In [ ]:
def XmeansCluster(embeddings_image, training_data, minClusters, maxClusters, maxIterations=3):
  clusterer = ee.Clusterer.wekaXMeans(minClusters=minClusters, maxClusters=maxClusters, distanceFunction="cosin").train(training_data)
  clustered = embeddings_image.cluster(clusterer)
  return clustered

In [ ]:
emb_XCluster = XmeansCluster(embeddings_mosaic, training, 4, 9)
mapa.addLayer(emb_XCluster.randomVisualizer().clip(geometry), {}, 'X Means Clusters')

### Clustering jerárquico aglomerativo de sklearn

# Add satellite imagery

In [ ]:
collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(geometry)
    .filterDate('2024-01-01', '2024-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

image = collection.median().clip(geometry) # .clip() is optional if you only want the exact shape

# 2. Define Visualization Parameters
# We select the Red (B4), Green (B3), and Blue (B2) bands for a true-color image
vis_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000,
    'gamma': 1.4 # Gamma correction to make the image look more natural
}

# 3. Add the Layers
# Add the satellite image first
mapa.addLayer(image, vis_params, 'Sentinel-2 True Color')

In [ ]:
display(mapa)

In [ ]:
mapa.add_basemap('Esri Satellite')

In [ ]:
mapa = geemap.Map()
mapa.setOptions('SATELLITE')
mapa.centerObject(geometry, 12)
mapa.add_basemap('Esri Satellite')